# 🎤 Whisper Speech-to-Text — Google Colab Demo

This notebook runs the **[docker-whisper](https://github.com/hwdsl2/docker-whisper)** API server directly in Colab (without Docker).

The server provides an **OpenAI-compatible** `POST /v1/audio/transcriptions` endpoint powered by [faster-whisper](https://github.com/SYSTRAN/faster-whisper).

**Quick start:** Select **Runtime → Run all** to run the full demo automatically using a built-in sample audio file.  
**Runtime recommendation:** Use **GPU** (Runtime → Change runtime type → T4 GPU) for faster inference. CPU works too.

## Step 1 — Install dependencies

In [ ]:
# Core server dependencies
!pip install -q faster-whisper fastapi "uvicorn[standard]" python-multipart
# PyAV provides bundled FFmpeg libraries for audio decoding
!pip install -q av
print('✅ Installation complete.')

## Step 2 — Download the API server script

In [ ]:
import urllib.request

API_SERVER_URL = "https://raw.githubusercontent.com/hwdsl2/docker-whisper/main/api_server.py"
urllib.request.urlretrieve(API_SERVER_URL, "/content/api_server.py")
print("✅ Downloaded api_server.py")

## Step 3 — Configure model & runtime settings

Edit the variables below to choose a different model or language.

| Model | Disk | RAM | Notes |
|---|---|---|---|
| `tiny` | ~75 MB | ~250 MB | Fastest, lower accuracy |
| `base` | ~145 MB | ~700 MB | Good balance — **default** |
| `small` | ~465 MB | ~1.5 GB | Better accuracy |
| `medium` | ~1.5 GB | ~5 GB | High accuracy |
| `large-v3-turbo` | ~1.6 GB | ~6 GB | Fast + high accuracy ⭐ |
| `large-v3` | ~3 GB | ~10 GB | Best accuracy |

In [ ]:
import os, subprocess

# ── Model settings ──────────────────────────────────────────────────────────
WHISPER_MODEL    = "base"  # tiny | base | small | medium | large-v3-turbo | large-v3
WHISPER_LANGUAGE = "auto"  # auto, or a BCP-47 code e.g. "en", "fr", "zh"

# ── Runtime settings ────────────────────────────────────────────────────────
# Detect GPU; fall back to CPU if unavailable.
try:
    _r = subprocess.run(["nvidia-smi"], capture_output=True, timeout=5)
    has_gpu = _r.returncode == 0
except Exception:
    has_gpu = False

# Use 'auto' so faster-whisper picks the best available device at runtime.
# On Colab GPU runtimes the CUDA libraries are pre-installed system-wide.
WHISPER_DEVICE       = "auto"
WHISPER_COMPUTE_TYPE = "float16" if has_gpu else "int8"
WHISPER_THREADS      = "4"    # CPU threads (ignored on CUDA)
WHISPER_PORT         = "9001"
WHISPER_BEAM         = "5"
WHISPER_API_KEY      = ""     # optional Bearer token; leave empty for open access

print(f"GPU available : {has_gpu}")
print(f"Device        : {WHISPER_DEVICE} ({'GPU' if has_gpu else 'CPU'} will be used)")
print(f"Model         : {WHISPER_MODEL}")
print(f"Compute type  : {WHISPER_COMPUTE_TYPE}")
print(f"Language      : {WHISPER_LANGUAGE}")

## Step 4 — Start the API server

In [ ]:
import os, subprocess, time, urllib.request, json

# Stop any previously running server instance before starting a new one
if "server_proc" in dir() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()

# Build environment for the server process
server_env = os.environ.copy()
server_env.update({
    "WHISPER_MODEL":        WHISPER_MODEL,
    "WHISPER_LANGUAGE":     WHISPER_LANGUAGE,
    "WHISPER_DEVICE":       WHISPER_DEVICE,
    "WHISPER_COMPUTE_TYPE": WHISPER_COMPUTE_TYPE,
    "WHISPER_THREADS":      WHISPER_THREADS,
    "WHISPER_PORT":         WHISPER_PORT,
    "WHISPER_BEAM":         WHISPER_BEAM,
    "WHISPER_API_KEY":      WHISPER_API_KEY,
    "HF_HOME":              "/content/whisper_cache",
})

os.makedirs("/content/whisper_cache", exist_ok=True)

log_file = open("/content/whisper_server.log", "w")
server_proc = subprocess.Popen(
    ["python", "api_server.py"],
    cwd="/content",
    env=server_env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

# Wait for the health endpoint to respond (model download + load can take a while)
health_url = f"http://127.0.0.1:{WHISPER_PORT}/health"
print(f"Starting server (pid {server_proc.pid}) — waiting for model to load…")
print("(The model will be downloaded from HuggingFace on first run — this may take a minute.)")

ready = False
for attempt in range(120):  # up to ~2 minutes
    time.sleep(2)
    try:
        with urllib.request.urlopen(health_url, timeout=3) as r:
            data = json.loads(r.read())
            print(f"\n✅ Server ready — model: {data.get('model')}")
            ready = True
            break
    except Exception:
        if attempt % 5 == 0:
            print(".", end="", flush=True)

if not ready:
    print("\n❌ Server did not become ready in time. Check logs:")
    with open("/content/whisper_server.log") as _f:
        print(_f.read()[-3000:])

## Step 5 — Transcribe audio

The cell below downloads a sample audio file automatically so you can try the API right away.
To use your own audio instead, uncomment and run the **Upload your own file** cell, then re-run the **Set up API connection** cell.

### Download sample audio

In [ ]:
import urllib.request

# Sample English speech audio (WAV, MIT License)
# Source: https://github.com/Azure-Samples/cognitive-services-speech-sdk
audio_path = "/content/sample_speech.wav"

urllib.request.urlretrieve(
    "https://github.com/Azure-Samples/cognitive-services-speech-sdk/raw/master/sampledata/audiofiles/katiesteve.wav",
    audio_path,
)
print(f"✅ Sample downloaded → {audio_path}")

### (Optional) Upload your own audio file instead

Run this cell manually to upload your own file. It will override the sample above.

In [ ]:
# NOTE: Run this cell manually only — it is intentionally skipped during 'Run all'.
# Uncomment the lines below and run this cell to upload your own audio file.

# from google.colab import files as colab_files
# print("Select an audio file to upload (mp3, wav, m4a, ogg, flac, webm, …)")
# uploaded = colab_files.upload()
# if uploaded:
#     audio_path = f"/content/{list(uploaded.keys())[0]}"
#     print(f"\nUploaded: {audio_path}")
# else:
#     print("No file uploaded — keeping previous audio_path.")

### Set up API connection

Run this cell once after choosing an audio file above.

In [ ]:
import os, requests, json

assert audio_path and os.path.exists(audio_path), \
    "No audio file found — run the sample download cell or upload cell above first."

API_BASE = f"http://127.0.0.1:{WHISPER_PORT}"

req_headers = {}
if WHISPER_API_KEY:
    req_headers["Authorization"] = f"Bearer {WHISPER_API_KEY}"

print(f"Audio file : {audio_path}")
print(f"API base   : {API_BASE}")

### Transcribe — JSON response (default)

In [ ]:
with open(audio_path, "rb") as f:
    response = requests.post(
        f"{API_BASE}/v1/audio/transcriptions",
        headers=req_headers,
        files={"file": (os.path.basename(audio_path), f)},
        data={"model": "whisper-1"},
    )

response.raise_for_status()
print("📝 Transcript:")
print(response.json()["text"])

### Transcribe — Verbose JSON (with timestamps and language detection)

In [ ]:
with open(audio_path, "rb") as f:
    response = requests.post(
        f"{API_BASE}/v1/audio/transcriptions",
        headers=req_headers,
        files={"file": (os.path.basename(audio_path), f)},
        data={"model": "whisper-1", "response_format": "verbose_json"},
    )

response.raise_for_status()
result = response.json()

print(f"Language detected : {result.get('language')} "
      f"(confidence {result.get('language_probability', 0):.1%})")
print(f"Duration          : {result.get('duration', 0):.2f}s")
print()
print("Segments:")
for seg in result.get("segments", []):
    print(f"  [{seg['start']:.2f}s → {seg['end']:.2f}s]  {seg['text']}")

### Transcribe — SRT subtitles

In [ ]:
with open(audio_path, "rb") as f:
    response = requests.post(
        f"{API_BASE}/v1/audio/transcriptions",
        headers=req_headers,
        files={"file": (os.path.basename(audio_path), f)},
        data={"model": "whisper-1", "response_format": "srt"},
    )

response.raise_for_status()
print(response.text)

### Transcribe — Streaming (SSE) — segments printed as they are decoded

In [ ]:
with open(audio_path, "rb") as f:
    with requests.post(
        f"{API_BASE}/v1/audio/transcriptions",
        headers=req_headers,
        files={"file": (os.path.basename(audio_path), f)},
        data={"model": "whisper-1", "stream": "true"},
        stream=True,
    ) as resp:
        resp.raise_for_status()
        buffer = ""
        for chunk in resp.iter_content(chunk_size=None):
            buffer += chunk.decode("utf-8", errors="replace")
            while "\n\n" in buffer:
                frame, buffer = buffer.split("\n\n", 1)
                if not frame.startswith("data: "):
                    continue
                event = json.loads(frame[6:])
                if event["type"] == "segment":
                    print(f"[{event['start']:.2f}s → {event['end']:.2f}s] {event['text']}")
                elif event["type"] == "done":
                    print(f"\n✅ Full transcript: {event['text']}")
                elif event["type"] == "error":
                    print(f"❌ Error: {event['detail']}")

## View server logs

In [ ]:
!tail -40 /content/whisper_server.log

## Stop the server

In [ ]:
if "server_proc" in dir() and server_proc.poll() is None:
    server_proc.terminate()
    try:
        server_proc.wait(timeout=10)
    except Exception:
        server_proc.kill()
    print("✅ Server stopped.")
else:
    print("ℹ️  Server is not running.")